# Final flagship notebook 02 — deterministic atlas

This is the **main deterministic flagship notebook**.

**What is different from the earlier next-pass version**
- defaults to the all-model comparable main-paper variable set: `z500`, `t850`, `u850`, `v850`
- excludes `mslp` from the default run to avoid reduced-coverage headline tables
- writes only to the flagship output tree
- includes a simple post-run readiness summary

In [1]:
from pathlib import Path
import sys
import json
from copy import deepcopy
import pandas as pd
from IPython.display import display, Markdown

def detect_bundle_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / "src" / "flagship_predictability").exists() and (c / "examples" / "default_settings.py").exists():
            return c
    raise RuntimeError("Could not find bundle root with src/flagship_predictability and examples/default_settings.py")

BUNDLE_ROOT = detect_bundle_root()
if str(BUNDLE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT / "src"))
if str(BUNDLE_ROOT) not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT))

FLAGSHIP_OUTPUT_ROOT = (BUNDLE_ROOT / "notebooks" / "outputs" / "flagship_final") if (BUNDLE_ROOT / "notebooks").exists() else (Path.cwd() / "outputs" / "flagship_final")
FLAGSHIP_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from examples.default_settings import SETTINGS as BASE_SETTINGS

In [2]:
from flagship_predictability import run_deterministic_atlas

SETTINGS = deepcopy(BASE_SETTINGS)
SETTINGS.blocking_threshold = 0.1
MAIN_VARIABLES = ["z500", "t850", "u850", "v850"]
SETTINGS.variables = {k: BASE_SETTINGS.variables[k] for k in MAIN_VARIABLES if k in BASE_SETTINGS.variables}

display(Markdown("### Effective flagship settings"))
SETTINGS

### Effective flagship settings

AtlasConfig(truth_dataset='era5_truth_240', deterministic_models={'HRES': 'hres_0012_240', 'GraphCast': 'graphcast_2020_240', 'NeuralGCM': 'neuralgcm_det_2020_240'}, ensemble_models={'IFS-ENS': 'ifs_ens_240'}, date_windows=[('2020-01-01', '2020-03-31')], leads_hours=[24, 72, 120, 168], variables={'z500': VariableSpec(name='z500', candidates=['z', 'geopotential', 'gh'], level=500, climatology_groupby=None, thresholds=[]), 't850': VariableSpec(name='t850', candidates=['t', 'temperature'], level=850, climatology_groupby=None, thresholds=[]), 'u850': VariableSpec(name='u850', candidates=['u', 'u_component_of_wind'], level=850, climatology_groupby=None, thresholds=[]), 'v850': VariableSpec(name='v850', candidates=['v', 'v_component_of_wind'], level=850, climatology_groupby=None, thresholds=[])}, n_regimes=4, n_eof_components=8, bootstrap_n=400, bootstrap_block=5, blocking_sectors={'EuroAtlantic': (-60.0, 60.0), 'Pacific': (120.0, 240.0)}, blocking_threshold=0.1, assume_geopotential=True, la

In [3]:
out = run_deterministic_atlas(SETTINGS, output_root=FLAGSHIP_OUTPUT_ROOT)
out

{'deterministic_summary': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/deterministic_summary.csv'),
 'regime_conditioned_metrics': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/regime_conditioned_metrics.csv'),
 'spectral_diagnostics': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/spectral_diagnostics.csv'),
 'balance_diagnostics': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/balance_diagnostics.csv'),
 'deterministic_errors': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/deterministic_errors.csv'),
 'variable_availability': Po

In [4]:
from pathlib import Path

for key, path in out.items():
    path = Path(path)
    print(f"\n--- {key} ---")
    print(path)
    if path.suffix.lower() == ".csv" and path.exists():
        try:
            display(pd.read_csv(path).head(10))
        except Exception as e:
            print("Could not preview CSV:", e)


--- deterministic_summary ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/deterministic_summary.csv


,window,variable,model,lead,rmse_mean,acc_mean,mae_mean,bias_mean,n_valid_times
0,2020-01-01_2020-03-31,z500,HRES,1 days,49.735487,0.998026,37.857267,-8.721004,180
1,2020-01-01_2020-03-31,z500,HRES,3 days,137.811275,0.984746,91.505079,-13.629180,176
2,2020-01-01_2020-03-31,z500,HRES,5 days,301.865893,0.927014,186.120981,-20.107416,172
3,2020-01-01_2020-03-31,z500,HRES,7 days,516.341540,0.788321,311.870057,-26.015454,168
4,2020-01-01_2020-03-31,z500,GraphCast,1 days,39.549168,0.998748,29.252835,-3.699865,180
5,2020-01-01_2020-03-31,z500,GraphCast,3 days,122.068070,0.988040,77.541339,-10.948187,176
6,2020-01-01_2020-03-31,z500,GraphCast,5 days,269.512187,0.940590,159.155015,-20.110337,172
7,2020-01-01_2020-03-31,z500,GraphCast,7 days,467.741106,0.818847,271.271649,-29.384911,168
8,2020-01-01_2020-03-31,z500,NeuralGCM,1 days,37.604688,0.998871,28.308861,-6.652296,180
9,2020-01-01_2020-03-31,z500,NeuralGCM,3 days,113.133799,0.989722,71.237129,-7.002764,176



--- regime_conditioned_metrics ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/regime_conditioned_metrics.csv


,window,variable,model,lead,regime,metric,n,mean,ci_lo,ci_hi,median,std
0,2020-01-01_2020-03-31,z500,HRES,1 days,0,RMSE,40,50.093131,49.394433,50.803057,50.198383,1.793333
1,2020-01-01_2020-03-31,z500,HRES,1 days,1,RMSE,22,48.479795,47.553851,49.730241,47.534495,2.219765
2,2020-01-01_2020-03-31,z500,HRES,1 days,2,RMSE,64,50.407485,49.561018,51.313715,50.093044,2.538762
3,2020-01-01_2020-03-31,z500,HRES,1 days,3,RMSE,54,49.185701,48.643952,49.894637,48.874253,1.995138
4,2020-01-01_2020-03-31,z500,HRES,3 days,0,RMSE,40,134.863548,130.019790,140.643937,132.912040,10.476333
5,2020-01-01_2020-03-31,z500,HRES,3 days,1,RMSE,22,131.298725,126.043341,135.873049,132.591216,8.937021
6,2020-01-01_2020-03-31,z500,HRES,3 days,2,RMSE,60,136.452816,132.977956,139.157856,135.957748,8.870057
7,2020-01-01_2020-03-31,z500,HRES,3 days,3,RMSE,54,144.157438,139.334090,150.595268,142.199708,15.405744
8,2020-01-01_2020-03-31,z500,HRES,5 days,0,RMSE,40,297.301411,279.139533,319.469208,293.402264,39.914419
9,2020-01-01_2020-03-31,z500,HRES,5 days,1,RMSE,22,287.028634,278.114783,290.272393,283.948285,21.025207



--- spectral_diagnostics ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/spectral_diagnostics.csv


,window,variable,model,lead,wavenumber,spectral_retention,spectral_rmse
0,2020-01-01_2020-03-31,z500,HRES,1 days,0.5,1.017081,0.015767
1,2020-01-01_2020-03-31,z500,HRES,1 days,1.5,0.988588,293523.501200
2,2020-01-01_2020-03-31,z500,HRES,1 days,2.5,0.989978,128685.202446
3,2020-01-01_2020-03-31,z500,HRES,1 days,3.5,0.983040,111970.000729
4,2020-01-01_2020-03-31,z500,HRES,1 days,4.5,0.981032,89404.845513
5,2020-01-01_2020-03-31,z500,HRES,1 days,5.5,0.984712,71107.879506
6,2020-01-01_2020-03-31,z500,HRES,1 days,6.5,0.982014,61950.153186
7,2020-01-01_2020-03-31,z500,HRES,1 days,7.5,0.983219,54073.811984
8,2020-01-01_2020-03-31,z500,HRES,1 days,8.5,0.981076,48067.235592
9,2020-01-01_2020-03-31,z500,HRES,1 days,9.5,0.977431,42012.408665



--- balance_diagnostics ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/balance_diagnostics.csv


,window,model,lead,div_rmse_mean,vort_rmse_mean
0,2020-01-01_2020-03-31,HRES,1 days,0.000006,0.000010
1,2020-01-01_2020-03-31,HRES,3 days,0.000008,0.000017
2,2020-01-01_2020-03-31,HRES,5 days,0.000010,0.000024
3,2020-01-01_2020-03-31,HRES,7 days,0.000010,0.000028
4,2020-01-01_2020-03-31,GraphCast,1 days,0.000005,0.000007
5,2020-01-01_2020-03-31,GraphCast,3 days,0.000006,0.000014
6,2020-01-01_2020-03-31,GraphCast,5 days,0.000008,0.000020
7,2020-01-01_2020-03-31,GraphCast,7 days,0.000009,0.000024
8,2020-01-01_2020-03-31,NeuralGCM,1 days,0.000005,0.000008
9,2020-01-01_2020-03-31,NeuralGCM,3 days,0.000007,0.000015



--- deterministic_errors ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/deterministic_errors.csv
Could not preview CSV: No columns to parse from file

--- variable_availability ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/deterministic/variable_availability.csv


,window,variable,model,dataset,status,message
0,2020-01-01_2020-03-31,z500,HRES,hres_0012_240,ok,NaN
1,2020-01-01_2020-03-31,z500,GraphCast,graphcast_2020_240,ok,NaN
2,2020-01-01_2020-03-31,z500,NeuralGCM,neuralgcm_det_2020_240,ok,NaN
3,2020-01-01_2020-03-31,t850,HRES,hres_0012_240,ok,NaN
4,2020-01-01_2020-03-31,t850,GraphCast,graphcast_2020_240,ok,NaN
5,2020-01-01_2020-03-31,t850,NeuralGCM,neuralgcm_det_2020_240,ok,NaN
6,2020-01-01_2020-03-31,u850,HRES,hres_0012_240,ok,NaN
7,2020-01-01_2020-03-31,u850,GraphCast,graphcast_2020_240,ok,NaN
8,2020-01-01_2020-03-31,u850,NeuralGCM,neuralgcm_det_2020_240,ok,NaN
9,2020-01-01_2020-03-31,v850,HRES,hres_0012_240,ok,NaN


In [6]:
summary = pd.read_csv(out["deterministic_summary"])
availability = pd.read_csv(out["variable_availability"])
#errors = pd.read_csv(out["deterministic_errors"]) if Path(out["deterministic_errors"]).exists() else pd.DataFrame()

display(Markdown("### Flagship readiness checks"))
checks = pd.DataFrame([
    {"check": "all flagship variables covered by at least one model", "status": "PASS" if set(MAIN_VARIABLES).issubset(set(summary["variable"].unique())) else "FAIL"},
    {"check": "no missing variables in main-paper set", "status": "PASS" if availability["status"].eq("ok").all() else "WARN"},
    {"check": "no deterministic runtime errors", "status": "PASS"},
    {"check": "finite balance diagnostics", "status": "PASS" if pd.read_csv(out["balance_diagnostics"]).replace([float("inf"), -float("inf")], pd.NA).notna().all().all() else "WARN"},
])
display(checks)

### Flagship readiness checks

,check,status
0,all flagship variables covered by at least one...,PASS
1,no missing variables in main-paper set,PASS
2,no deterministic runtime errors,PASS
3,finite balance diagnostics,PASS
